# Ориентация текстовых кропов: 0° или 180°

Для каждого кропа с текстом нужно предсказать `p_180` — вероятность того, что текст перевёрнут. Метрика — `1 − Brier score`, при равном качестве лучше более быстрая и компактная модель.

## Подход

**Данные.** Обучающей выборки в задании нет, поэтому метки получаем сами: берём заведомо ровный текст и половину примеров поворачиваем на 180°. Источники:
- **синтетика** — строки из русской и английской Википедии и шаблоны, типичные для объявлений (телефоны, цены, артикулы, модели), отрендеренные ~70 семействами Google Fonts с кириллицей; фон, тени, обводка, дуги, куски соседних строк, размытие, JPEG — всё подобрано под то, что видно в тесте;
- **реальные кропы** — фото русских вывесок, рекламы и ценников с Wikimedia Commons (открытые лицензии), нарезанные детектором текста PP-OCRv4 так же, как нарезаны тестовые кропы.

**Модели: дистилляция.**
- Учитель — ConvNeXt-Tiny (27.8 млн параметров, вход 48 px по высоте), дообучен с ImageNet на смеси синтетики и реальных кропов.
- Студент — PP-LCNet ×0.5 (0.6 млн параметров, вход 32 px), учится на смеси откалиброванных вероятностей учителя и истинных меток. В проде работает только он.
- Вход обеих сетей — серый холст фиксированного размера: кроп по центру, длинные строки сжимаются по горизонтали (признаки ориентации вертикальные и при этом сохраняются).

**Калибровка.** Brier сильно штрафует уверенные ошибки, поэтому:
- антисимметричный TTA: `logit = (f(x) − f(rot180 x)) / 2`, откуда `p(x) + p(rot180 x) = 1` точно;
- температура, подобранная на валидации под Brier.

**Валидация.** Реальные кропы Commons, отложенные по исходному фото (кропы одного фото не попадают одновременно в обучение и валидацию). Температура оценивается кросс-фиттингом на двух фолдах. Дополнительно считаем score с весами, выравнивающими распределение высоты и пропорций кропов с тестом. Сравниваем с константой 0.5 и готовым классификатором ориентации PaddleOCR.

**Результаты и что пробовали** — в разделе 9.

**Open-source:** PyTorch, timm (веса ImageNet), RapidOCR (модели PaddleOCR: детектор PP-OCRv4, классификатор ориентации — как бейзлайн), ONNX Runtime, Google Fonts (OFL), корпуса Leipzig Corpora Collection (Википедия, CC BY-SA), фото Wikimedia Commons (CC BY / CC BY-SA / CC0; авторы и лицензии — в `data/commons_manifest.csv`).

**Разделы:** 0. Окружение · 1. Режим запуска · 2. Тестовые данные · 3. Данные для обучения · 4. Учитель · 5. Студент · 6. Валидация и калибровка · 7. Скорость и размер · 8. Submission · 9. Итоги

## 0. Окружение

Ячейка ниже создаёт `.venv` в корне репозитория, ставит зависимости из `requirements.txt` и регистрирует ядро Jupyter «Python (avito-orient)». Её можно запускать из любого ядра с Python 3.10+.

После неё выберите ядро `.venv` (VS Code: Select Kernel → Python Environments → `.venv`) и перезапустите notebook: следующие ячейки работают уже в нём.

In [ ]:
import os
import sys
from pathlib import Path

VENV_PY = Path(".venv") / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

if not VENV_PY.exists():
    !"{sys.executable}" -m venv .venv
!"{VENV_PY}" -m pip install -q -r requirements.txt
!"{VENV_PY}" -m ipykernel install --user --name avito-orient --display-name "Python (avito-orient)"

In [ ]:
import json
import sys
from pathlib import Path

if Path(sys.prefix).resolve() != (Path.cwd() / ".venv").resolve():
    raise RuntimeError(f"Ядро не из .venv: {sys.executable}. Выберите ядро .venv и перезапустите notebook.")

sys.path.insert(0, str(Path.cwd() / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

from orient import download, infer, pipeline as P, viz
from orient.metrics import report, sigmoid
from orient.models import STUDENT, TEACHER
from orient.paths import DATA, TEST_IMAGES

## 1. Режим запуска

- `RUN_TRAINING = False` (по умолчанию) — обучение пропускается, берутся веса студента и калибровка из `artifacts/`. Весь notebook проходит за несколько минут и воспроизводит `submission.csv`.
- `RUN_TRAINING = True` — полный путь с нуля: скачивание фото и весов ImageNet, нарезка кропов, обучение учителя (~2 ч на Apple M4 Pro) и студента, оценка. Уже готовые этапы (`checkpoints/`, `data/crops/`) не пересчитываются; чтобы переобучить, удалите их.

Тестовые картинки ожидаются в `test/test/images/`; другой путь можно задать переменной окружения `ORIENT_TEST` до запуска ядра.

In [ ]:
RUN_TRAINING = False

cfg = P.Config()
print(f"тест: {TEST_IMAGES} ({len(infer.test_paths(TEST_IMAGES))} картинок)")
pd.Series(vars(cfg))

## 2. Тестовые данные

Что видно в тесте и на что опираемся при подготовке данных:
- строки горизонтальные: вертикальных кропов почти нет, значит повороты на 90°/270° не рассматриваем;
- кропы мелкие, есть совсем нечитаемые — модель должна уметь выдавать неуверенные вероятности;
- в основном кириллица, плюс латиница, цифры и артикулы, рукописный текст, скриншоты, вывески, логотипы;
- в кроп часто попадают куски соседних строк сверху или снизу.

In [ ]:
test_paths = infer.test_paths(TEST_IMAGES)
test_wh = P.image_sizes(test_paths)
sizes = pd.DataFrame({"высота": test_wh[:, 1], "ширина / высота": test_wh[:, 0] / test_wh[:, 1]})
display(sizes.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1).T)

rng = np.random.default_rng(0)
sample = [Image.open(test_paths[i]) for i in rng.choice(len(test_paths), 24, replace=False)]
viz.contact_sheet(sample)

## 3. Данные для обучения

### 3.1 Скачивание

Шрифты, корпуса и список фото Commons лежат в репозитории (`data/`). Скачиваются только сами фото Commons по списку и веса ImageNet — это нужно лишь для обучения.

In [ ]:
if RUN_TRAINING:
    download.download_commons(download.commons_manifest())
    download.download_pretrained()
download.summary()

### 3.2 Синтетика

Каждый пример однозначно задаётся номером, поэтому набор воспроизводим. Ниже — ровные примеры (до поворота).

In [ ]:
rend = P.renderer()
print(f"семейств шрифтов с кириллицей: {len(rend.families)}")
viz.contact_sheet([rend(np.random.default_rng([1, i])) for i in range(24)])

### 3.3 Реальные кропы

Детектор PP-OCRv4 режет фото так же, как нарезаны тестовые кропы. Вертикальные боксы пропускаем: их ориентация не определена. Разбиение train/val — по хэшу имени фото.

In [ ]:
meta = P.prepare_crops(cfg) if RUN_TRAINING or (P.CROPS / "crops.csv").exists() else None
if meta is not None and len(meta):
    display(meta.groupby("split").agg(кропов=("crop_id", "size"), фото=("photo_id", "nunique")))
    shown = meta.sample(24, random_state=0)
    display(viz.contact_sheet([Image.open(P.CROPS / "images" / f"{c}.png") for c in shown.crop_id]))
else:
    print("Кропы не нарезаны: этот шаг нужен только для обучения (RUN_TRAINING = True).")

## 4. Учитель

ConvNeXt-Tiny с весами ImageNet (первая свёртка сведена к одному каналу), вход 48 × 480. AdamW, косинусный шаг обучения с разогревом. Цель — метка поворота. Контроль качества во время обучения — подвыборка реальной валидации.

In [ ]:
if RUN_TRAINING:
    P.train_teacher(cfg)
viz.plot_logs({"учитель": P.ARTIFACTS / "teacher_log.json"});

## 5. Студент

PP-LCNet ×0.5, вход 32 × 320. На каждом шаге учитель (с антисимметричным TTA и своей температурой) размечает тот же батч в разрешении 48 px, цель студента — `alpha · p_учителя + (1 − alpha) · метка`. Мягкие метки передают студенту неуверенность учителя на неоднозначных кропах, что напрямую улучшает Brier.

In [ ]:
if RUN_TRAINING:
    P.train_student(cfg)
viz.plot_logs({"учитель": P.ARTIFACTS / "teacher_log.json", "студент": P.ARTIFACTS / "student_log.json"});

## 6. Валидация и калибровка

Полная валидация: реальные кропы Commons (`real`) и отдельный синтетический набор (`synth`). Для каждой модели — без калибровки и с температурой, подобранной кросс-фиттингом. `score_test_weighted` — score с весами, выравнивающими высоту и пропорции кропов с тестом.

In [ ]:
if RUN_TRAINING:
    P.evaluate(cfg)
metrics = pd.read_csv(P.ARTIFACTS / "metrics.csv")
display(metrics.round(4))
calib = json.loads((P.ARTIFACTS / "calibration.json").read_text())
calib

In [ ]:
preds = pd.read_csv(P.ARTIFACTS / "val_predictions.csv")
main = preds[preds.set == calib["fitted_on"]]
y = main.y.to_numpy()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
viz.plot_reliability({"студент": sigmoid(main.z_student.to_numpy()),
                      "студент + T": sigmoid(main.z_student.to_numpy() / calib["temperature"]),
                      "RapidOCR cls": main.p_rapidocr.to_numpy()}, y, ax=ax[0])
ax[0].set_title("Калибровка")

# Качество студента по высоте кропа: где модель ошибается.
p = sigmoid(main.z_student.to_numpy() / calib["temperature"])
bins = pd.cut(main.h, [0, 16, 24, 36, 54, 80, 1000])
by_h = pd.DataFrame({"h": bins, "err": (p - y) ** 2}).groupby("h", observed=True).err.agg(["mean", "size"])
ax[1].bar(by_h.index.astype(str), 1 - by_h["mean"])
ax[1].set_ylim(0.5, 1)
ax[1].set_title("1 − Brier по высоте кропа, px")
plt.tight_layout()

## 7. Скорость и размер

Студент экспортируется в ONNX вместе с TTA и температурой (вход — холст, выход — `p_180`). Время замеряется в ONNX Runtime на CPU отдельно для подготовки холста и для сети.

In [ ]:
bench = P.benchmark()
bench.round(3)

## 8. Submission

Инференс на CPU с детерминированными алгоритмами: при повторном запуске получается тот же файл.

In [ ]:
sub = P.make_submission(Path("submission.csv"))
print(sub.shape, "среднее p_180:", round(sub.p_180.mean(), 4))
display(sub.head())
sub.p_180.hist(bins=50);

## 9. Итоги

Заполняется по результатам экспериментов.